<a href="https://colab.research.google.com/github/ehiabh1/turbofan.rul/blob/main/02_rf_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
if not os.path.exists("/content/turbofan.rul"):
    !git clone https://github.com/ehiabh1/turbofan.rul.git
%cd /content/turbofan.rul
!git pull

/content/turbofan.rul
Already up to date.


In [2]:
!wget -q https://data.nasa.gov/docs/legacy/CMAPSSData.zip
!unzip -o -q CMAPSSData.zip -d cmapss_raw
!find cmapss_raw -name "*_FD001.txt" -exec cp {} data/ \;
!ls data

README.md  RUL_FD001.txt  test_FD001.txt  train_FD001.txt


In [3]:
from src.load_data import load_fd
train, test, rul = load_fd("FD001")
train.shape, test.shape

((20631, 27), (13096, 27))

In [4]:
from src.preprocess import drop_dead, clip_rul
t = clip_rul(drop_dead(train))
print(t.shape)
print(t["RUL"].max())
print(train["RUL"].max())

(20631, 17)
125
361


In [8]:
from sklearn.ensemble import RandomForestRegressor



FEATURES = [c for c in clip_rul(drop_dead(train)).columns if c not in ("unit", "cycle", "RUL")]
print(len(FEATURES), FEATURES)

tr = clip_rul(drop_dead(train))
te = clip_rul(drop_dead(test))

X_train, y_train = tr[FEATURES], tr["RUL"]

# Test engines are scored at their last recorded cycle only.
te_last = te.groupby("unit").tail(1)
X_test, y_test = te_last[FEATURES], te_last["RUL"]

print(X_train.shape, X_test.shape)   # expect (20631, 14) and (100, 14)

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

14 ['s2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']
(20631, 14) (100, 14)


RandomForestRegressor(n_jobs=-1, random_state=42)

In [10]:
from sklearn.metrics import mean_squared_error
import numpy as np
preds = rf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(rmse)

17.186845842096798


In [11]:
naive = np.full(len(y_test), y_train.mean())
print(np.sqrt(mean_squared_error(y_test, naive)))

41.94179565195834
